In [0]:
# MVP Engenharia de Dados
# ETAPA 02 - Diagnóstico de qualidade e preparação da camada Silver

from pyspark.sql import functions as F

BRONZE_TABLE = "workspace.mvp_bronze.student_productivity_raw"
SILVER_TABLE = "workspace.mvp_silver.student_productivity_clean"

df = spark.table(BRONZE_TABLE)

print("=== CAMADA BRONZE CARREGADA ===")
print("Registros:", df.count())
print("Colunas:", len(df.columns))

display(df.limit(10))

=== CAMADA BRONZE CARREGADA ===
Registros: 5999
Colunas: 20


student_id,age,gender,study_hours_per_day,sleep_hours,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,exercise_minutes,assignments_completed,attendance_percentage,stress_level,focus_score,final_grade,productivity_score,_ingestion_ts,_source_file
1,23,Female,4.35,3.63,3.38,2.73,1.83,5.26,6,347,111,2,57.21,10,57,81.87,33.78,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
2,20,Male,6.14,6.58,5.48,1.51,3.13,1.73,13,403,28,10,91.27,10,49,60.9,48.99,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
3,29,Female,4.98,3.26,4.83,3.63,0.18,4.71,1,419,102,8,63.14,2,38,86.22,36.6,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
4,27,Female,3.19,4.58,10.06,3.95,5.75,2.52,9,178,28,18,40.51,6,50,71.77,19.87,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
5,24,Male,7.67,6.21,3.02,1.59,5.46,5.65,8,436,105,7,45.53,6,41,90.13,52.9,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
6,29,Other,7.18,3.52,4.02,3.74,1.42,0.16,10,392,12,3,47.58,10,70,59.48,47.31,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
7,21,Female,9.06,6.36,11.45,5.99,2.2,4.44,14,87,28,15,43.5,8,35,62.71,41.23,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
8,23,Female,6.37,4.86,3.31,1.37,4.36,5.13,2,152,103,17,75.22,6,59,52.22,53.81,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
9,26,Male,4.19,4.87,9.66,2.87,0.1,3.38,13,460,42,11,44.79,3,39,76.15,25.99,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
10,19,Female,7.28,9.56,2.13,0.81,1.35,2.55,7,416,107,6,79.15,10,73,88.53,73.18,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv


In [0]:
# Verificação de unicidade do identificador student_id

total_registros = df.count()
ids_distintos = df.select("student_id").distinct().count()

duplicados = total_registros - ids_distintos

print("=== ANÁLISE DE UNICIDADE ===")
print("Total de registros:", total_registros)
print("student_id distintos:", ids_distintos)
print("Possíveis duplicatas:", duplicados)

=== ANÁLISE DE UNICIDADE ===
Total de registros: 5999
student_id distintos: 5999
Possíveis duplicatas: 0


In [0]:
# Verificação de completude por atributo

print("=== ANÁLISE DE COMPLETUDE ===")

colunas_originais = [
    c for c in df.columns
    if not c.startswith("_")
]

resultado_nulos = []

for c in colunas_originais:
    nulos = df.filter(
        F.col(c).isNull() |
        (F.trim(F.col(c)) == "")
    ).count()

    percentual = (nulos / total_registros) * 100

    resultado_nulos.append(
        (c, nulos, round(percentual, 2))
    )

df_nulos = spark.createDataFrame(
    resultado_nulos,
    ["coluna", "valores_nulos", "percentual_nulos"]
)

display(df_nulos)

=== ANÁLISE DE COMPLETUDE ===


coluna,valores_nulos,percentual_nulos
student_id,0,0.0
age,0,0.0
gender,0,0.0
study_hours_per_day,0,0.0
sleep_hours,0,0.0
phone_usage_hours,0,0.0
social_media_hours,0,0.0
youtube_hours,0,0.0
gaming_hours,0,0.0
breaks_per_day,0,0.0


In [0]:
# Conversão dos tipos para preparação da camada Silver

integer_cols = [
    "student_id",
    "age",
    "breaks_per_day",
    "coffee_intake_mg",
    "exercise_minutes",
    "assignments_completed",
    "stress_level",
    "focus_score"
]

double_cols = [
    "study_hours_per_day",
    "sleep_hours",
    "phone_usage_hours",
    "social_media_hours",
    "youtube_hours",
    "gaming_hours",
    "attendance_percentage",
    "final_grade",
    "productivity_score"
]

df_typed = df

for c in integer_cols:
    df_typed = df_typed.withColumn(
        c, F.col(c).cast("integer")
    )

for c in double_cols:
    df_typed = df_typed.withColumn(
        c, F.col(c).cast("double")
    )

# Padronização da variável categórica
df_typed = df_typed.withColumn(
    "gender",
    F.initcap(F.trim(F.col("gender")))
)

print("=== TIPAGEM CONCLUÍDA ===")
df_typed.printSchema()

=== TIPAGEM CONCLUÍDA ===
root
 |-- student_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- study_hours_per_day: double (nullable = true)
 |-- sleep_hours: double (nullable = true)
 |-- phone_usage_hours: double (nullable = true)
 |-- social_media_hours: double (nullable = true)
 |-- youtube_hours: double (nullable = true)
 |-- gaming_hours: double (nullable = true)
 |-- breaks_per_day: integer (nullable = true)
 |-- coffee_intake_mg: integer (nullable = true)
 |-- exercise_minutes: integer (nullable = true)
 |-- assignments_completed: integer (nullable = true)
 |-- attendance_percentage: double (nullable = true)
 |-- stress_level: integer (nullable = true)
 |-- focus_score: integer (nullable = true)
 |-- final_grade: double (nullable = true)
 |-- productivity_score: double (nullable = true)
 |-- _ingestion_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [0]:
# Validação após conversão dos tipos

print("=== NULOS APÓS TIPAGEM ===")

for c in integer_cols + double_cols:
    nulos = df_typed.filter(F.col(c).isNull()).count()
    print(f"{c}: {nulos}")

=== NULOS APÓS TIPAGEM ===
student_id: 0
age: 0
breaks_per_day: 0
coffee_intake_mg: 0
exercise_minutes: 0
assignments_completed: 0
stress_level: 0
focus_score: 0
study_hours_per_day: 0
sleep_hours: 0
phone_usage_hours: 0
social_media_hours: 0
youtube_hours: 0
gaming_hours: 0
attendance_percentage: 0
final_grade: 0
productivity_score: 0


In [0]:
# Verificação de valores mínimos e máximos
# para avaliar consistência e plausibilidade

numeric_cols = integer_cols + double_cols

resultado_dominios = []

for c in numeric_cols:
    stats = (
        df_typed
        .agg(
            F.min(c).alias("minimo"),
            F.max(c).alias("maximo")
        )
        .first()
    )

    resultado_dominios.append(
        (c, float(stats["minimo"]), float(stats["maximo"]))
    )

df_dominios = spark.createDataFrame(
    resultado_dominios,
    ["variavel", "minimo", "maximo"]
)

display(df_dominios)

variavel,minimo,maximo
student_id,1.0,5999.0
age,17.0,29.0
breaks_per_day,1.0,14.0
coffee_intake_mg,0.0,499.0
exercise_minutes,0.0,119.0
assignments_completed,0.0,19.0
stress_level,1.0,10.0
focus_score,30.0,99.0
study_hours_per_day,0.5,10.0
sleep_hours,3.0,10.0


In [0]:
# Validação específica das principais variáveis do MVP

df_typed.select(
    F.min("final_grade").alias("nota_minima"),
    F.max("final_grade").alias("nota_maxima"),
    F.min("productivity_score").alias("produtividade_minima"),
    F.max("productivity_score").alias("produtividade_maxima")
).show()

print("=== CATEGORIAS DE GÊNERO ===")
df_typed.groupBy("gender").count().orderBy("gender").show()

+-----------+-----------+--------------------+--------------------+
|nota_minima|nota_maxima|produtividade_minima|produtividade_maxima|
+-----------+-----------+--------------------+--------------------+
|       40.0|      99.99|                1.91|                99.9|
+-----------+-----------+--------------------+--------------------+

=== CATEGORIAS DE GÊNERO ===
+------+-----+
|gender|count|
+------+-----+
|Female| 2868|
|  Male| 2897|
| Other|  234|
+------+-----+



In [0]:
# Construção da camada Silver

df_silver = (
    df_typed
    .withColumn(
        "entertainment_hours",
        F.round(
            F.col("social_media_hours")
            + F.col("youtube_hours")
            + F.col("gaming_hours"),
            2
        )
    )
)

print("=== SILVER PREPARADA ===")
print("Registros:", df_silver.count())
print("Colunas:", len(df_silver.columns))

display(df_silver.limit(10))

=== SILVER PREPARADA ===
Registros: 5999
Colunas: 21


student_id,age,gender,study_hours_per_day,sleep_hours,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,exercise_minutes,assignments_completed,attendance_percentage,stress_level,focus_score,final_grade,productivity_score,_ingestion_ts,_source_file,entertainment_hours
1,23,Female,4.35,3.63,3.38,2.73,1.83,5.26,6,347,111,2,57.21,10,57,81.87,33.78,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,9.82
2,20,Male,6.14,6.58,5.48,1.51,3.13,1.73,13,403,28,10,91.27,10,49,60.9,48.99,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,6.37
3,29,Female,4.98,3.26,4.83,3.63,0.18,4.71,1,419,102,8,63.14,2,38,86.22,36.6,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,8.52
4,27,Female,3.19,4.58,10.06,3.95,5.75,2.52,9,178,28,18,40.51,6,50,71.77,19.87,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,12.22
5,24,Male,7.67,6.21,3.02,1.59,5.46,5.65,8,436,105,7,45.53,6,41,90.13,52.9,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,12.7
6,29,Other,7.18,3.52,4.02,3.74,1.42,0.16,10,392,12,3,47.58,10,70,59.48,47.31,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,5.32
7,21,Female,9.06,6.36,11.45,5.99,2.2,4.44,14,87,28,15,43.5,8,35,62.71,41.23,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,12.63
8,23,Female,6.37,4.86,3.31,1.37,4.36,5.13,2,152,103,17,75.22,6,59,52.22,53.81,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,10.86
9,26,Male,4.19,4.87,9.66,2.87,0.1,3.38,13,460,42,11,44.79,3,39,76.15,25.99,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,6.35
10,19,Female,7.28,9.56,2.13,0.81,1.35,2.55,7,416,107,6,79.15,10,73,88.53,73.18,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,4.71


In [0]:
SILVER_TABLE = "workspace.mvp_silver.student_productivity_clean"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print("Tabela Silver persistida com sucesso:")
print(SILVER_TABLE)

Tabela Silver persistida com sucesso:
workspace.mvp_silver.student_productivity_clean


In [0]:
df_silver_check = spark.table(SILVER_TABLE)

print("=== VALIDAÇÃO DA SILVER ===")
print("Tabela:", SILVER_TABLE)
print("Registros:", df_silver_check.count())
print("Colunas:", len(df_silver_check.columns))

print("\nSchema:")
df_silver_check.printSchema()

display(df_silver_check.limit(10))

=== VALIDAÇÃO DA SILVER ===
Tabela: workspace.mvp_silver.student_productivity_clean
Registros: 5999
Colunas: 21

Schema:
root
 |-- student_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- study_hours_per_day: double (nullable = true)
 |-- sleep_hours: double (nullable = true)
 |-- phone_usage_hours: double (nullable = true)
 |-- social_media_hours: double (nullable = true)
 |-- youtube_hours: double (nullable = true)
 |-- gaming_hours: double (nullable = true)
 |-- breaks_per_day: integer (nullable = true)
 |-- coffee_intake_mg: integer (nullable = true)
 |-- exercise_minutes: integer (nullable = true)
 |-- assignments_completed: integer (nullable = true)
 |-- attendance_percentage: double (nullable = true)
 |-- stress_level: integer (nullable = true)
 |-- focus_score: integer (nullable = true)
 |-- final_grade: double (nullable = true)
 |-- productivity_score: double (nullable = true)
 |-- _ingestion_ts: timestamp (nullable =

student_id,age,gender,study_hours_per_day,sleep_hours,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,exercise_minutes,assignments_completed,attendance_percentage,stress_level,focus_score,final_grade,productivity_score,_ingestion_ts,_source_file,entertainment_hours
1,23,Female,4.35,3.63,3.38,2.73,1.83,5.26,6,347,111,2,57.21,10,57,81.87,33.78,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,9.82
2,20,Male,6.14,6.58,5.48,1.51,3.13,1.73,13,403,28,10,91.27,10,49,60.9,48.99,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,6.37
3,29,Female,4.98,3.26,4.83,3.63,0.18,4.71,1,419,102,8,63.14,2,38,86.22,36.6,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,8.52
4,27,Female,3.19,4.58,10.06,3.95,5.75,2.52,9,178,28,18,40.51,6,50,71.77,19.87,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,12.22
5,24,Male,7.67,6.21,3.02,1.59,5.46,5.65,8,436,105,7,45.53,6,41,90.13,52.9,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,12.7
6,29,Other,7.18,3.52,4.02,3.74,1.42,0.16,10,392,12,3,47.58,10,70,59.48,47.31,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,5.32
7,21,Female,9.06,6.36,11.45,5.99,2.2,4.44,14,87,28,15,43.5,8,35,62.71,41.23,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,12.63
8,23,Female,6.37,4.86,3.31,1.37,4.36,5.13,2,152,103,17,75.22,6,59,52.22,53.81,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,10.86
9,26,Male,4.19,4.87,9.66,2.87,0.1,3.38,13,460,42,11,44.79,3,39,76.15,25.99,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,6.35
10,19,Female,7.28,9.56,2.13,0.81,1.35,2.55,7,416,107,6,79.15,10,73,88.53,73.18,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv,4.71


In [0]:
# Resumo final de qualidade dos dados
# Célula autônoma: pode ser executada independentemente das anteriores.

from pyspark.sql import functions as F

BRONZE_TABLE = "workspace.mvp_bronze.student_productivity_raw"
QUALITY_TABLE = "workspace.mvp_silver.data_quality_summary"

# Carrega novamente a Bronze
df_quality_source = spark.table(BRONZE_TABLE)

# Considera somente os 18 atributos originais
colunas_originais = [
    c for c in df_quality_source.columns
    if not c.startswith("_")
]

total_registros = df_quality_source.count()

resultado_qualidade = []

for c in colunas_originais:

    nulos = (
        df_quality_source
        .filter(
            F.col(c).isNull() |
            (F.trim(F.col(c)) == "")
        )
        .count()
    )

    distintos = (
        df_quality_source
        .select(c)
        .distinct()
        .count()
    )

    percentual_nulos = round(
        (nulos / total_registros) * 100,
        2
    )

    resultado_qualidade.append(
        (
            c,
            total_registros,
            nulos,
            percentual_nulos,
            distintos
        )
    )

df_quality = spark.createDataFrame(
    resultado_qualidade,
    [
        "atributo",
        "total_registros",
        "valores_nulos",
        "percentual_nulos",
        "valores_distintos"
    ]
)

print("=== RESUMO DE QUALIDADE DOS DADOS ===")
print("Total de registros:", total_registros)
print("Total de atributos analisados:", len(colunas_originais))

display(df_quality)

=== RESUMO DE QUALIDADE DOS DADOS ===
Total de registros: 5999
Total de atributos analisados: 18


atributo,total_registros,valores_nulos,percentual_nulos,valores_distintos
student_id,5999,0,0.0,5999
age,5999,0,0.0,13
gender,5999,0,0.0,3
study_hours_per_day,5999,0,0.0,951
sleep_hours,5999,0,0.0,701
phone_usage_hours,5999,0,0.0,1144
social_media_hours,5999,0,0.0,801
youtube_hours,5999,0,0.0,601
gaming_hours,5999,0,0.0,601
breaks_per_day,5999,0,0.0,14


In [0]:
# Persistência do resumo de qualidade na camada Silver

QUALITY_TABLE = "workspace.mvp_silver.data_quality_summary"

(
    df_quality.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUALITY_TABLE)
)

print("Tabela de qualidade persistida com sucesso:")
print(QUALITY_TABLE)

Tabela de qualidade persistida com sucesso:
workspace.mvp_silver.data_quality_summary


In [0]:
# Validação da tabela de qualidade

df_quality_check = spark.table(QUALITY_TABLE)

print("=== VALIDAÇÃO DA TABELA DE QUALIDADE ===")
print("Tabela:", QUALITY_TABLE)
print("Registros:", df_quality_check.count())
print("Colunas:", len(df_quality_check.columns))

display(df_quality_check)

=== VALIDAÇÃO DA TABELA DE QUALIDADE ===
Tabela: workspace.mvp_silver.data_quality_summary
Registros: 18
Colunas: 5


atributo,total_registros,valores_nulos,percentual_nulos,valores_distintos
student_id,5999,0,0.0,5999
age,5999,0,0.0,13
gender,5999,0,0.0,3
study_hours_per_day,5999,0,0.0,951
sleep_hours,5999,0,0.0,701
phone_usage_hours,5999,0,0.0,1144
social_media_hours,5999,0,0.0,801
youtube_hours,5999,0,0.0,601
gaming_hours,5999,0,0.0,601
breaks_per_day,5999,0,0.0,14
